# Cosine LR Scheduler with Warmup

**难度:** Medium

实现带线性预热的余弦学习率调度。

该调度器在预热阶段线性增加学习率，之后按余弦曲线衰减，广泛用于 Transformer 训练。

**签名:** `cosine_lr_schedule(step, total_steps, warmup_steps, max_lr, min_lr=0.0) -> float`

**参数:**
- `step` — 当前训练步数
- `total_steps` — 总步数
- `warmup_steps` — 预热步数
- `max_lr`, `min_lr` — 峰值和最小学习率

**返回:** 浮点数学习率

**约束:**
- 预热阶段：从 0 线性增加到 max_lr
- 衰减阶段：`min_lr + 0.5*(max_lr-min_lr)*(1+cos(pi*progress))`

**提示:**
1. step < warmup_steps → lr = max_lr * step / warmup_steps
2. step >= total_steps → lr = min_lr
3. progress = (step - warmup_steps) / (total_steps - warmup_steps)
   → min_lr + 0.5*(max_lr-min_lr)*(1 + cos(π·progress))

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [ ]:
# ✅ SOLUTION

# ---- Step 1: 预热阶段 ----
def cosine_lr_schedule(step, total_steps, warmup_steps, max_lr, min_lr=0.0):
    # 预热阶段：学习率从 0 线性增长到 max_lr
    # 例如 warmup_steps=100，step=50 时 lr = max_lr * 50/100 = 0.5*max_lr
    if step < warmup_steps:
        return max_lr * step / warmup_steps

    # ---- Step 2: 训练结束后保持最小学习率 ----
    if step >= total_steps:
        return min_lr

    # ---- Step 3: 余弦衰减阶段 ----
    # progress ∈ [0, 1]：从预热结束到训练结束的进度
    progress = (step - warmup_steps) / (total_steps - warmup_steps)

    # 余弦退火公式：lr = min_lr + 0.5*(max_lr-min_lr)*(1+cos(π*progress))
    # 当 progress=0: cos(0)=1   → lr = max_lr（刚预热完，保持最大学习率）
    # 当 progress=1: cos(π)=-1  → lr = min_lr（训练结束，降到最小）
    # 中间过程：余弦曲线平滑下降，比线性衰减更"柔和"
    return min_lr + 0.5 * (max_lr - min_lr) * (1.0 + math.cos(math.pi * progress))

In [ ]:
# Demo
lrs = [cosine_lr_schedule(i, 100, 10, 0.001) for i in range(101)]
print(f'Start: {lrs[0]:.6f}, Warmup end: {lrs[10]:.6f}, Mid: {lrs[55]:.6f}, End: {lrs[100]:.6f}')

In [ ]:
from torch_judge import check
check('cosine_lr')

## 📝 核心思路总结

1. **Warmup 防止初期不稳定**：训练初期梯度方差大，线性预热让学习率从 0 慢慢增大
2. **余弦衰减比线性更好**：开始衰减快、后期衰减慢，给模型更多时间在最优附近微调
3. **progress 从 0→1 映射到 cos 从 1→-1**：lr 从 max_lr 平滑降到 min_lr